# V3：既往合并症 ICD embedding

三组消融：static_lr → burden_lr → embedding_lr。

burden 包括既往住院次数、不同确切码数、诊断事件数和有无既往诊断；前三个用log1p压缩。embedding_lr 在 burden_lr 上增加 TF-IDF + TruncatedSVD（最多32维），以区分“诊断更丰富”与“潜在表示”带来的增益。

词表、IDF、SVD和缩放只在训练子集中拟合，验证集只 transform。没历史、空词表、仅一个词均有保留患者的回退逻辑。未知验证码不会增加词表。

这属于结构化诊断表示，不是 clinical notes；当前没有临床文本数据。静态 CLIN 共病仍继承 V3 的既往∪索引慢性编码假设，新增 embedding 完全使用既往已结束住院。后续部署仍需说明原静态编码在24小时时是否可得。

**balanced 简化计算版**：固定基模型参数、full保留5×10外层验证；LASSO内部5折/30λ，stacking内部3折。与上级严格版使用独立缓存；前6折旧预测不混入。当前没有运行新版本真实患者模型。

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# 从扩展目录或 notebooks 子目录定位项目，避免硬编码 Windows 盘符。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dn_extensions" / "settings.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请从 dn_mortality_repro_V3_extensions 或其 notebooks 子目录启动。")
sys.path.insert(0, str(ROOT))
print("扩展目录：", ROOT)


In [ ]:
# pilot 是1×3外层小规模试跑；正式实验将 PROFILE 改为 "full"。
# 仅把 RUN_EXPERIMENT 改成 True 才会启动；full可能需要较长时间。
PROFILE = "pilot"
RUN_EXPERIMENT = False
EXPERIMENTS = ['static_lr', 'burden_lr', 'embedding_lr']
if RUN_EXPERIMENT:
    from dn_extensions.evaluation import run_evaluation
    from dn_extensions.settings import profile_options
    output, summary, comparisons = run_evaluation(profile_options(PROFILE), EXPERIMENTS)
    display(summary)
    display(comparisons)
else:
    print("尚未启动本Notebook中的模型训练。")


In [ ]:
# 优先查看正式 full 结果；没有结果时打印状态，不重新训练。
RESULT_PROFILE = "full"
latest = ROOT / "results" / RESULT_PROFILE / "latest.txt"
if latest.exists():
    folder = latest.parent / latest.read_text().strip()
    summary = pd.read_csv(folder / "summary.csv")
    display(summary[summary.experiment.isin(EXPERIMENTS)])
    print("详细记录：", folder / "report.md")
else:
    print("该模式尚无实验结果。pilot 结果需把 RESULT_PROFILE 改为 pilot 才显示。")


## 实验状态与结论（自动更新）

运行模式：`full`；方法：balanced 固定参数；结果目录：`results/full/27464ba98305edbc`。

LASSO 5折/30λ、stacking 3折；静态与扩展使用同一简化方法。严格版的部分折没有混入。

固定留出集未参与本实验。树模型扩展不需要 stacking 内部交叉拟合；原固定参数及训练折内预处理不变。

Holm 校正覆盖当前已有配对比较；后续增加实验时，校正 p 值可能变化。

```text

   experiment  n_outer_folds  AUROC_fold_mean  AUROC_fold_sd  AUPRC_fold_mean  Brier_fold_mean  calibration_slope_fold_mean  seconds_total  AUROC_pooled_mean  AUPRC_pooled_mean  Brier_pooled_mean  cal_intercept_pooled_mean  cal_slope_pooled_mean
    burden_lr             50          0.79936        0.02666          0.60660          0.15087                      0.91420      578.69442            0.79925            0.59466            0.15086                    0.00359                0.89227
    burden_rf             50          0.79724        0.02871          0.60343          0.15232                      1.22278       28.27988            0.79581            0.58966            0.15232                    0.00337                1.18404
   burden_xgb             50          0.79396        0.03022          0.60583          0.15265                      0.93979        6.26590            0.79309            0.59511            0.15264                    0.00653                0.91576
 embedding_lr             50          0.79591        0.02741          0.60562          0.15229                      0.85161      582.82427            0.79555            0.59278            0.15228                    0.00813                0.83040
 embedding_rf             50          0.79965        0.02798          0.60840          0.15269                      1.39541       32.57728            0.79825            0.59542            0.15268                    0.00490                1.35224
embedding_xgb             50          0.79429        0.02760          0.60649          0.15232                      0.97897        9.29661            0.79331            0.59430            0.15231                   -0.00756                0.95188
    static_lr             50          0.79928        0.02648          0.60740          0.15092                      0.92073      502.87958            0.79885            0.59482            0.15091                    0.00385                0.89831
    static_rf             50          0.79798        0.02809          0.60618          0.15177                      1.19841       15.22184            0.79671            0.59435            0.15177                    0.00179                1.16211
   static_xgb             50          0.79504        0.02965          0.60572          0.15250                      0.94083        4.61352            0.79408            0.59662            0.15249                    0.00766                0.91690

```

配对差异：

```text

   experiment  reference metric  mean_diff  sd_diff  se_corrected       lo      hi        t  df  p_value  k  r  p_holm_current_family
    static_rf  static_lr  AUROC   -0.00130  0.01457       0.00528 -0.01190 0.00931 -0.24552  49  0.80708 10  5                    1.0
    static_rf  static_lr  AUPRC   -0.00122  0.02900       0.01050 -0.02232 0.01989 -0.11569  49  0.90837 10  5                    1.0
    static_rf  static_lr  Brier    0.00085  0.00469       0.00170 -0.00256 0.00427  0.50207  49  0.61787 10  5                    1.0
   static_xgb  static_lr  AUROC   -0.00423  0.01409       0.00510 -0.01449 0.00602 -0.82968  49  0.41074 10  5                    1.0
   static_xgb  static_lr  AUPRC   -0.00167  0.02894       0.01048 -0.02273 0.01938 -0.15972  49  0.87376 10  5                    1.0
   static_xgb  static_lr  Brier    0.00158  0.00519       0.00188 -0.00220 0.00536  0.83926  49  0.40540 10  5                    1.0
    burden_lr  static_lr  AUROC    0.00008  0.00315       0.00114 -0.00221 0.00237  0.07071  49  0.94392 10  5                    1.0
    burden_lr  static_lr  AUPRC   -0.00080  0.00669       0.00242 -0.00566 0.00407 -0.32854  49  0.74391 10  5                    1.0
    burden_lr  static_lr  Brier   -0.00005  0.00109       0.00039 -0.00084 0.00074 -0.12687  49  0.89956 10  5                    1.0
 embedding_lr  burden_lr  AUROC   -0.00345  0.01003       0.00363 -0.01075 0.00385 -0.94986  49  0.34685 10  5                    1.0
 embedding_lr  burden_lr  AUPRC   -0.00098  0.01402       0.00508 -0.01118 0.00922 -0.19368  49  0.84723 10  5                    1.0
 embedding_lr  burden_lr  Brier    0.00142  0.00345       0.00125 -0.00109 0.00393  1.13866  49  0.26038 10  5                    1.0
    burden_rf  static_rf  AUROC   -0.00074  0.00616       0.00223 -0.00522 0.00374 -0.33253  49  0.74091 10  5                    1.0
    burden_rf  static_rf  AUPRC   -0.00275  0.01287       0.00466 -0.01212 0.00661 -0.59075  49  0.55740 10  5                    1.0
    burden_rf  static_rf  Brier    0.00055  0.00173       0.00063 -0.00071 0.00181  0.87819  49  0.38412 10  5                    1.0
   burden_xgb static_xgb  AUROC   -0.00108  0.00676       0.00245 -0.00600 0.00384 -0.44121  49  0.66100 10  5                    1.0
   burden_xgb static_xgb  AUPRC    0.00010  0.01749       0.00633 -0.01263 0.01283  0.01602  49  0.98728 10  5                    1.0
   burden_xgb static_xgb  Brier    0.00015  0.00214       0.00077 -0.00140 0.00171  0.19740  49  0.84433 10  5                    1.0
 embedding_rf  burden_rf  AUROC    0.00241  0.00885       0.00320 -0.00403 0.00885  0.75282  49  0.45516 10  5                    1.0
 embedding_rf  burden_rf  AUPRC    0.00497  0.01607       0.00582 -0.00673 0.01666  0.85397  49  0.39728 10  5                    1.0
 embedding_rf  burden_rf  Brier    0.00037  0.00239       0.00087 -0.00137 0.00211  0.42331  49  0.67392 10  5                    1.0
embedding_xgb burden_xgb  AUROC    0.00033  0.01064       0.00385 -0.00741 0.00807  0.08550  49  0.93221 10  5                    1.0
embedding_xgb burden_xgb  AUPRC    0.00067  0.02335       0.00846 -0.01632 0.01766  0.07910  49  0.93728 10  5                    1.0
embedding_xgb burden_xgb  Brier   -0.00033  0.00395       0.00143 -0.00321 0.00254 -0.23263  49  0.81702 10  5                    1.0
 embedding_rf  static_rf  AUROC    0.00167  0.00963       0.00349 -0.00534 0.00868  0.47943  49  0.63377 10  5                    1.0
 embedding_rf  static_rf  AUPRC    0.00222  0.01939       0.00702 -0.01189 0.01632  0.31582  49  0.75348 10  5                    1.0
 embedding_rf  static_rf  Brier    0.00092  0.00260       0.00094 -0.00098 0.00281  0.97278  49  0.33544 10  5                    1.0
embedding_xgb static_xgb  AUROC   -0.00075  0.00988       0.00358 -0.00794 0.00644 -0.20974  49  0.83474 10  5                    1.0
embedding_xgb static_xgb  AUPRC    0.00077  0.02388       0.00865 -0.01661 0.01815  0.08909  49  0.92938 10  5                    1.0
embedding_xgb static_xgb  Brier   -0.00018  0.00389       0.00141 -0.00301 0.00265 -0.12793  49  0.89873 10  5                    1.0

```

- `static_rf` 相对 `static_lr`：ΔAUROC=-0.0013，校正95% CI [-0.0119, 0.0093]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `static_xgb` 相对 `static_lr`：ΔAUROC=-0.0042，校正95% CI [-0.0145, 0.0060]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `burden_lr` 相对 `static_lr`：ΔAUROC=0.0001，校正95% CI [-0.0022, 0.0024]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `embedding_lr` 相对 `burden_lr`：ΔAUROC=-0.0035，校正95% CI [-0.0108, 0.0038]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `burden_rf` 相对 `static_rf`：ΔAUROC=-0.0007，校正95% CI [-0.0052, 0.0037]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `burden_xgb` 相对 `static_xgb`：ΔAUROC=-0.0011，校正95% CI [-0.0060, 0.0038]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `embedding_rf` 相对 `burden_rf`：ΔAUROC=0.0024，校正95% CI [-0.0040, 0.0089]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `embedding_xgb` 相对 `burden_xgb`：ΔAUROC=0.0003，校正95% CI [-0.0074, 0.0081]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `embedding_rf` 相对 `static_rf`：ΔAUROC=0.0017，校正95% CI [-0.0053, 0.0087]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `embedding_xgb` 相对 `static_xgb`：ΔAUROC=-0.0008，校正95% CI [-0.0079, 0.0064]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

## 2026-10-14 合并症消融复核

static_lr、burden_lr、embedding_lr各完成5×10个外层折。burden衡量既往诊断数量和住院次数；embedding在burden上增加既往ICD的TF-IDF/SVD表示，均在训练子集拟合。

| 模型 | AUROC折均值 | AUPRC合并OOF均值 | Brier合并OOF均值 | 校准斜率合并OOF均值 |
|---|---:|---:|---:|---:|
| static_lr | 0.7993 | 0.5948 | 0.1509 | 0.898 |
| burden_lr | 0.7994 | 0.5947 | 0.1509 | 0.892 |
| embedding_lr | 0.7959 | 0.5928 | 0.1523 | 0.830 |

- `burden_lr` 相对 `static_lr`：ΔAUROC=0.0001，Nadeau–Bengio校正95% CI [-0.0022, 0.0024]，p=0.944，Holm校正p=1.000。
- `embedding_lr` 相对 `burden_lr`：ΔAUROC=-0.0035，Nadeau–Bengio校正95% CI [-0.0108, 0.0038]，p=0.347，Holm校正p=1.000。

**诊断数量特征几乎不改变性能，当前embedding没有显示增益。** 五次重复的embedding合并OOF AUROC均低于burden对照，Brier和校准斜率的点估计也较差；但差异的校正区间跨0，不能写成显著损害预测能力。

实施核查：50折均生成32维embedding，没有空词表导致整组表示被跳过的问题。不能据此断言所有合并症或文本表示都无效；本实验仅针对既往住院ICD、当前TF-IDF/SVD维数及固定参数LR。

现阶段保留静态模型和stacking作为主要候选，将本消融作为完整的阴性结果报告。联合stacking尚未完成；它可以作为原计划的最后一项预设探索，但不能提前期待一定提高性能。
